# Importing HTML Data with Pandas

Pandas is capable of reading and writing data in a variety of file formats, including HTML. A list of all data formats that pandas supports is located [here](https://pandas.pydata.org/pandas-docs/stable/user_guide/io.html).


In [ ]:
import pandas as pd


## HTML

Reading in html can be useful if there is a table of data on a website that you want to parse. We'll use the [2024 1040 Intructions]( https://www.irs.gov/instructions/i1040gi ) from the IRS.

In [ ]:
url = "https://www.irs.gov/instructions/i1040gi"
tax_tables = pd.read_html(url)
len(tax_tables) # tax_tables contains all of the tables on this webpage


In [ ]:
for index, table in enumerate(tax_tables):
    print(f"== {index}")
    print(table.columns)
    print()


### Locate tax table

In [ ]:
tax_tables[173]


In [ ]:
tax_table = tax_tables[173]
tax_table


### Rename columns in tax table

In [ ]:
tax_table.columns = "Low High Single Married MFJ HoH".split()


In [ ]:
tax_table


### Clean table

The table is segmented into 10,000, 20,000, 30,0000, etc.  These aren't actual values for the table, but rather dividers between income levels.  Remove segment rows.


In [ ]:
segments = ( tax_table['High'] == tax_table["MFJ"] )
drop_indexes = tax_table[segments].index
tax_table_clean = tax_table.drop( drop_indexes ).dropna(axis = "rows")
tax_table_clean


In [ ]:
tax_table_clean.info()


In [ ]:
for col in tax_table_clean.columns:
  tax_table_clean[col] = tax_table_clean[col].astype(int)


In [ ]:
tax_table_clean.info()


In [ ]:
df = tax_table_clean
df


In [ ]:
filter = ( (df["Low"] >= 1400) & (df["High"] <= 1600 ) )
df[filter]


### Plot Tax Due versus Income

In [ ]:
# Create the plot
subplot = df.plot(x='High', y='Single', kind='line')

# Add title and labels
subplot.set_title('Tax vs Income')
subplot.set_xlabel('Income')
subplot.set_ylabel('Tax')
;


### Identify tax bracket changes

#### Calculate midpoint between High and Low income values

In [ ]:
df["midpoint"] = ( df["High"] + df["Low"] ) / 2
df


#### Calculate tax as a percentage of income

Percent = tax due / income * 100




In [ ]:
df["pct"] = df["Single"].div(df["midpoint"]).mul(100)
df


#### Calculate diff in percentage

In [ ]:
df["pct_delta"] = df["pct"].diff()
df


#### Calculate diff of diff

In [ ]:
df["pct_delta_delta"] = df["pct_delta"].diff().mul(10_000)
df


In [ ]:
# n = 200
# df[n:n+50]
low = ( df["High"] > 11_650 )
high = ( df["High"] < 12_200 )

df[ low & high ]


In [ ]:
filter = ( df["pct_delta_delta"] > 1 ) & ( df["Low"] > 3000 )
df[ filter ]


From [Marginal Tax Rates 2025]( https://www.irs.gov/newsroom/irs-releases-tax-inflation-adjustments-for-tax-year-2025#:~:text=Marginal%20rates.%20For%20tax%20year%202025 )

Historical info: [Marginal Tax Rates]( https://www.irs.gov/newsroom/inflation-adjusted-tax-items-by-tax-year )

- 37% for incomes over \$626,350 (\$751,600 for married couples filing jointly)
- 35% for incomes over \$250,525 (\$501,050 for married couples filing jointly).
- 32% for incomes over \$197,300 (\$394,600 for married couples filing jointly).
- 24% for incomes over \$103,350 (\$206,700 for married couples filing jointly).
- 22% for incomes over \$48,475 (\$96,950 for married couples filing jointly).
- 12% for incomes over \$11,925 (\$23,850 for married couples filing jointly).
- 10% for incomes over \$0 (\$0 for married couples filing jointly)


In [ ]:
import matplotlib.pyplot as plt

subplot = df.plot(
  figsize = (12, 6),
  kind = "line",
  x = "High",
  y = "pct",
  title = "Marginal Tax vs AGI",
  xlabel = "Adjusted Gross Income ($)",
  ylabel = "Marginal Tax (%)",
  legend = False,
  # ylim = [9.5, 10.4],
  # xlim = [10500, 12000],
)

x_positions = [
  0,
  11_925,
  48_475,
  103_350,
]

for x in x_positions:
  subplot.axvline(x=x, color='r', linestyle='--', linewidth=1)

y_positions = [
  0,
  10,
  12,
  22,
  24,
]

for y in y_positions:
  subplot.axhline(y=y, color='r', linestyle='--', linewidth=1)

